# CrisisLens Libya — Unified MARBERT Multi-task Model

**Capstone Notebook**

This notebook implements one unified MARBERT model for Arabic/Libyan crisis reports using a **shared encoder and three task-specific heads**:

1. **Location extraction (NER):** `O / B-LOC / I-LOC`
2. **Incident category classification:** 6 classes
3. **Priority classification:** `Low / Medium / High / Critical`

The NER head starts from the previously fine-tuned **IDRISI MARBERT** checkpoint, while the category and priority heads are trained jointly on the Libya-specific Balaagh dataset.

## 1. Overview

The objective is to process one crisis report with one shared MARBERT encoder and return three outputs: explicit location mentions, incident category, and incident priority.

**Task:** Unified multi-task learning: token classification + two sequence-classification tasks.  
**Model / Method:** MARBERT with a shared encoder and three output heads.

All three losses back-propagate through the same encoder during each training step.

> The category and priority labels are single-choice classes. Therefore this is multi-task learning with two **multi-class** classifiers plus NER, not true multi-label classification.

### Environment Setup

In [31]:
%pip -q install "transformers>=4.46,<5" "safetensors>=0.4,<1" "scikit-learn>=1.3,<2" "pandas>=2,<3"

Note: you may need to restart the kernel to use updated packages.


## 2. Dataset

The experiment uses the **Balaagh CrisisLens Libya dataset** with fixed `train`, `val`, and `test` splits.

Main columns:
- `report`: report text.
- `incident_type`: one of six incident categories.
- `priority`: Low, Medium, High, or Critical.
- `location_json`: character-span location annotations.
- `source`: real or generated.

Executed experiment:

| Split | Reports | Real reports | Gold LOC mentions |
|---|---:|---:|---:|
| Train | 826 | 222 | 746 |
| Validation | 118 | 33 | 103 |
| Test | 236 | 62 | 210 |

The validation split is used for model selection. The held-out test split is used only after training.

### Configuration

In [32]:
from pathlib import Path
from collections import Counter
import os, io, re, csv, json, zipfile, random, hashlib, shutil
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from safetensors.torch import save_file, load_file
from IPython.display import display

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type!='cuda':
    raise RuntimeError(' GPU missed ')

OUT=Path('/kaggle/working/crisislens_unified_marbert')
OUT.mkdir(parents=True,exist_ok=True)
ROOTS=[Path('/kaggle/input'),Path('/kaggle/working')]
MANUAL_DATA_PATH=''        # optional: path to balaagh_dataset_ner_ready.zip or directory with 3 CSVs
MANUAL_IDRISI_PATH=''      # optional: path to original IDRISI NER final_model or its ZIP
MAX_LEN=192
EPOCHS=8
TRAIN_BATCH=8
EVAL_BATCH=16
GRAD_ACCUM=2
ENCODER_LR=8e-6
HEAD_LR=2e-5
WEIGHT_DECAY=0.01
WARMUP_RATIO=0.1
PATIENCE=2
# Relative influence on the shared encoder; tune on VALIDATION only
LOSS_WEIGHTS={'ner':1.0,'category':1.0,'priority':1.0}
# Composite selection uses validation entity F1 and both validation macro-F1s
SELECT_WEIGHTS={'ner':0.40,'category':0.30,'priority':0.30}
CATEGORIES=['Fire/Explosion','Flood/Severe Weather','Infrastructure/Utilities',
            'Road/Transportation','People at Risk/Medical','Other']
PRIORITIES=['Low','Medium','High','Critical']
NER_TAGS=['O','B-LOC','I-LOC']
CAT2ID={x:i for i,x in enumerate(CATEGORIES)}
PRI2ID={x:i for i,x in enumerate(PRIORITIES)}
TAG2ID={x:i for i,x in enumerate(NER_TAGS)}
assert abs(sum(SELECT_WEIGHTS.values())-1)<1e-8

Device: cuda


### Dataset Loading

In [33]:
def matches(name):
    return sorted({p for root in ROOTS if root.exists() for p in root.rglob(name) if p.is_file()},key=str)

def safe_extract(zippath, dest):
    dest.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(zippath) as z:
        for member in z.infolist():
            target=(dest/member.filename).resolve()
            if not target.is_relative_to(dest.resolve()):
                raise ValueError('Unsafe ZIP member: '+member.filename)
        z.extractall(dest)
    return dest

def read_csv_splits(folder):
    return {s:pd.read_csv(folder/(s+'.csv'),encoding='utf-8-sig',keep_default_na=False)
            for s in ('train','val','test')}

if MANUAL_DATA_PATH:
    candidate=Path(MANUAL_DATA_PATH)
else:
    candidates=matches('balaagh_dataset_ner_ready.zip')
    if candidates:
        candidate=candidates[0]
    else:
        folders=sorted({q.parent for q in matches('train.csv')
                        if (q.parent/'val.csv').is_file() and (q.parent/'test.csv').is_file()},key=str)
        if len(folders)!=1:
            print('Candidate folders:',folders)
            raise FileNotFoundError('حددي MANUAL_DATA_PATH: لم نعثر على Balaagh dataset واحدة واضحة.')
        candidate=folders[0]
if candidate.suffix.lower()=='.zip':
    folder=safe_extract(candidate,OUT/'input_data')
    folders=[q.parent for q in folder.rglob('train.csv')
             if (q.parent/'val.csv').is_file() and (q.parent/'test.csv').is_file()]
    if len(folders)!=1: raise ValueError('ZIP must contain exactly one train/val/test CSV set: '+str(folders))
    folder=folders[0]
else:
    folder=candidate
SPLITS=read_csv_splits(folder)
REQUIRED={'report','incident_type','priority','location_json','source'}
for split,df in SPLITS.items():
    if not REQUIRED.issubset(df.columns):
        raise ValueError(f'{split}: missing columns {REQUIRED-set(df.columns)}')
    if df.empty: raise ValueError(f'{split} is empty')
    print(split,len(df),'real:',int(df.source.astype(str).str.lower().eq('real').sum()))
print('Data folder:',folder)

train 826 real: 222
val 118 real: 33
test 236 real: 62
Data folder: /kaggle/input/datasets/ritaj3li3enabdallah/balag/balaagh_dataset


### Dataset Overview

The original split structure is preserved. No new random split is created. Results are reported on both the full test set and the subset of 62 reports marked as real, because generated/template examples may not reflect all real-world linguistic variation.

## 3. Data Preprocessing

Preprocessing consists of:

1. Normalizing priority capitalization (`LOW`, `low`, `Low` → `Low`).
2. Parsing and validating `location_json`.
3. Verifying each location with `text[start:end]`.
4. Applying one explicit known correction for the `شاطئ القويعة طرابلس` annotation.
5. Rejecting overlapping spans that cannot be represented by flat BIO.
6. Checking for identical text leakage across dataset splits.
7. Loading the original IDRISI NER checkpoint and its fast tokenizer.
8. Converting character spans to `B-LOC / I-LOC / O` using `offset_mapping`.
9. Assigning `-100` to special/padding positions so they do not contribute to the NER loss.
10. Stopping if a gold location is truncated by `MAX_LEN`.

### Normalize Priority Labels

In [34]:
# Normalize priority labels in ALL splits, not just the last dataframe.
priority_map = {name.casefold(): name for name in PRIORITIES}
for split, frame in SPLITS.items():
    original = frame['priority'].astype(str)
    normalized = original.str.strip().str.casefold().map(priority_map)
    unknown = sorted(set(original[normalized.isna()]))
    if unknown:
        raise ValueError(f'{split}: unrecognized priority labels: {unknown}')
    changed = int((original != normalized).sum())
    frame['priority'] = normalized
    print(f'{split}: standardized {changed} priority labels')


train: standardized 69 priority labels
val: standardized 12 priority labels
test: standardized 22 priority labels


### Validate Location Annotations and Split Integrity

In [35]:
def parse_entities(text,payload,split,row_num):
    try: data=json.loads(payload) if isinstance(payload,str) and payload.strip() else []
    except Exception as exc: raise ValueError(f'{split} row {row_num}: malformed JSON: {exc}')
    if not isinstance(data,list): raise ValueError(f'{split} row {row_num}: location_json must be a list')
    result=[]
    for e in data:
        st,en,tx,lab=e.get('start'),e.get('end'),e.get('text'),e.get('label')
        if not(isinstance(st,int) and isinstance(en,int) and 0<=st<en<=len(text) and lab=='LOC'):
            raise ValueError(f'{split} row {row_num}: invalid entity {e}')
        if text[st:en] != tx:
            normalized = re.sub(r"\s+", " ", tx).strip()

            if normalized == "شاطئ القويعة في طرابلس":
                matches = list(
                    re.finditer(
                        r"شاطئ\s+القويعة\s+طرابلس",
                        text
                    )
                )

                if len(matches) == 1:
                    st, en = matches[0].span()
                    tx = text[st:en]

                    print(
                        f"Fixed {split} row {row_num}: "
                        f"{tx!r} -> ({st}, {en})"
                    )
                else:
                    raise ValueError(
                        f"{split} row {row_num}: "
                        "Could not uniquely locate the corrected place."
                    )
            else:
                raise ValueError(
                    f"{split} row {row_num}: "
                    f"bad span {e}, actual={text[st:en]!r}"
                )
        result.append({'start':st,'end':en,'text':tx,'label':'LOC'})
    result.sort(key=lambda x:(x['start'],x['end']))
    if any(b['start']<a['end'] for a,b in zip(result,result[1:])):
        raise ValueError(f'{split} row {row_num}: overlapping entities not supported in flat BIO')
    return result

for split,df in SPLITS.items():
    for col,allowed in [('incident_type',CATEGORIES),('priority',PRIORITIES)]:
        unknown=set(df[col])-set(allowed)
        if unknown:raise ValueError(f'{split}: unexpected {col} labels: {unknown}')
    df['entities']=[parse_entities(str(text),payload,split,i+2)
                    for i,(text,payload) in enumerate(zip(df.report,df.location_json))]
    print(split,'gold LOC:',sum(map(len,df.entities)))

# Flag, don't quietly drop, identical texts appearing across partitions.
def normalized_hash(text):
    return hashlib.sha1(re.sub(r'\s+',' ',str(text)).strip().encode('utf-8')).hexdigest()
for a,b in [('train','val'),('train','test'),('val','test')]:
    intersection=set(SPLITS[a].report.map(normalized_hash)) & set(SPLITS[b].report.map(normalized_hash))
    if intersection:raise ValueError(f'Exact normalized text leakage across {a}/{b}: {len(intersection)}')
print('QA passed; no identical cross-split texts.')

Fixed train row 671: 'شاطئ القويعة طرابلس' -> (18, 37)
train gold LOC: 746
val gold LOC: 103
test gold LOC: 210
QA passed; no identical cross-split texts.


### Load Original IDRISI NER Checkpoint and Tokenizer

In [36]:
def is_idrisi_ner(path):
    if not path.is_dir() or not (path/'config.json').is_file():return False
    if not any((path/x).exists() for x in ('model.safetensors','pytorch_model.bin',
                                            'model.safetensors.index.json','pytorch_model.bin.index.json')):return False
    try:
        cfg=json.loads((path/'config.json').read_text(encoding='utf-8'))
        labels=set(str(x) for x in cfg.get('id2label',{}).values())
        return {'O','B-LOC','I-LOC'}.issubset(labels)
    except Exception:return False

if MANUAL_IDRISI_PATH:
    p=Path(MANUAL_IDRISI_PATH)
    if p.suffix.lower()=='.zip':
        tmp=safe_extract(p,OUT/'unpacked_original_idrisi')
        paths=[x.parent for x in tmp.rglob('config.json') if is_idrisi_ner(x.parent)]
        finals=[x for x in paths if x.name=='final_model']
        paths=finals or paths
        if len(paths)!=1:raise ValueError('Multiple IDRISI models in ZIP; set exact model directory')
        IDRISI_PATH=paths[0]
    else: IDRISI_PATH=p
else:
    zips=matches('marbert_idrisi_typeless_model.zip')
    if len(zips)>1:raise ValueError('Multiple original IDRISI ZIPs; set MANUAL_IDRISI_PATH')
    if zips:
        tmp=safe_extract(zips[0],OUT/'unpacked_original_idrisi')
        paths=[x.parent for x in tmp.rglob('config.json') if is_idrisi_ner(x.parent)]
    else:
        paths=[x.parent for root in ROOTS if root.exists() for x in root.rglob('config.json')
               if is_idrisi_ner(x.parent) and not any(t in str(x).lower() for t in
               ('three_tasks','unified','plus_balaagh','balaagh_model','balag-edris'))]
    finals=[x for x in paths if x.name=='final_model']
    paths=finals or paths
    if len(paths)!=1:
        print('Candidates:',*[str(x) for x in paths],sep='\n - ')
        raise FileNotFoundError('حددي MANUAL_IDRISI_PATH لمسار original IDRISI NER final_model.')
    IDRISI_PATH=paths[0]
if not is_idrisi_ner(IDRISI_PATH):raise ValueError('Selected path is not an IDRISI BIO NER checkpoint')
TOKENIZER=AutoTokenizer.from_pretrained(str(IDRISI_PATH),use_fast=True)
if not TOKENIZER.is_fast:raise RuntimeError('Fast tokenizer needed for offsets')
print('Original model:',IDRISI_PATH)
print('Tokenizer:',type(TOKENIZER).__name__)

Original model: /kaggle/working/crisislens_unified_marbert/unpacked_original_idrisi/final_model
Tokenizer: BertTokenizerFast


### Tokenization and Joint Labels

Each report is tokenized once. The same `input_ids` and attention mask are paired with:
- token-level NER labels,
- one category label,
- one priority label.

In [37]:
class UnifiedDataset(Dataset):
    def __init__(self,df,name):
        self.df=df.reset_index(drop=True)
        self.data=[]
        lost=0
        for i,r in self.df.iterrows():
            text=str(r['report']);entities=r['entities']
            enc=TOKENIZER(text,max_length=MAX_LEN,truncation=True,padding='max_length',
                          return_offsets_mapping=True,return_tensors=None)
            offsets=enc.pop('offset_mapping')
            labels=[];covered=[False]*len(entities);previous=-1
            for st,en in offsets:
                if st==en:
                    labels.append(-100);continue
                match=[j for j,e in enumerate(entities) if st<e['end'] and en>e['start']]
                if len(match)>1:raise ValueError(f'{name} row {i}: overlapping token with two LOCs')
                if not match:labels.append(TAG2ID['O']);previous=-1
                else:
                    j=match[0]; covered[j]=True
                    labels.append(TAG2ID['I-LOC'] if previous==j else TAG2ID['B-LOC'])
                    previous=j
            last=max([en for st,en in offsets if en>st],default=0)
            if not all(covered) or any(e['end']>last for e in entities):
                lost+=1
                raise ValueError(f'{name} row {i}: LOC truncated by MAX_LEN={MAX_LEN}; increase MAX_LEN')
            item={'input_ids':enc['input_ids'],'attention_mask':enc['attention_mask'],
                  'ner_labels':labels,'category_labels':CAT2ID[r['incident_type']],
                  'priority_labels':PRI2ID[r['priority']]}
            if 'token_type_ids' in enc: item['token_type_ids']=enc['token_type_ids']
            self.data.append({k:torch.tensor(v,dtype=torch.long) for k,v in item.items()})
        print(name,':',len(self.data),'rows,',sum(map(len,self.df.entities)),'aligned entities; truncated:',lost)
    def __len__(self):return len(self.data)
    def __getitem__(self,i):return self.data[i]

DATA={name:UnifiedDataset(frame,name) for name,frame in SPLITS.items()}
TRAIN_LOADER=DataLoader(DATA['train'],batch_size=TRAIN_BATCH,shuffle=True,pin_memory=True)
VAL_LOADER=DataLoader(DATA['val'],batch_size=EVAL_BATCH,shuffle=False,pin_memory=True)
TEST_LOADER=DataLoader(DATA['test'],batch_size=EVAL_BATCH,shuffle=False,pin_memory=True)
print('Batches:',len(TRAIN_LOADER),len(VAL_LOADER),len(TEST_LOADER))

train : 826 rows, 746 aligned entities; truncated: 0
val : 118 rows, 103 aligned entities; truncated: 0
test : 236 rows, 210 aligned entities; truncated: 0
Batches: 104 8 15


## 4. Model / Method

The architecture contains one shared MARBERT encoder and three heads.

### NER head
A linear token-classification head predicts `O`, `B-LOC`, or `I-LOC`. Its weights are initialized from the original IDRISI-trained NER checkpoint.

### Incident-category head
Uses the first-token representation and predicts one of:
- Fire/Explosion
- Flood/Severe Weather
- Infrastructure/Utilities
- Road/Transportation
- People at Risk/Medical
- Other

### Priority head
Uses the same shared representation and predicts:
- Low
- Medium
- High
- Critical

The executed model contained **162,260,749 parameters**.

### Joint Loss

`L_total = 1.0 L_NER + 1.0 L_category + 1.0 L_priority`

Thus, all three tasks update the shared MARBERT encoder.

In [38]:
class UnifiedMARBERT(nn.Module):
    def __init__(self,ner_ckpt):
        super().__init__()
        old=AutoModelForTokenClassification.from_pretrained(str(ner_ckpt))
        self.encoder=old.base_model
        hidden=int(self.encoder.config.hidden_size)
        self.dropout=nn.Dropout(0.1)
        self.ner_head=nn.Linear(hidden,len(NER_TAGS))
        self.category_head=nn.Linear(hidden,len(CATEGORIES))
        self.priority_head=nn.Linear(hidden,len(PRIORITIES))
        old_mapping={str(v):int(k) for k,v in old.config.id2label.items()}
        with torch.no_grad():
            for name,new_idx in TAG2ID.items():
                if name not in old_mapping:raise ValueError('NER checkpoint missing '+name)
                old_idx=old_mapping[name]
                self.ner_head.weight[new_idx].copy_(old.classifier.weight[old_idx])
                self.ner_head.bias[new_idx].copy_(old.classifier.bias[old_idx])
        self.encoder.config.return_dict=True
        del old
    def forward(self,input_ids,attention_mask,token_type_ids=None):
        kwargs={'input_ids':input_ids,'attention_mask':attention_mask,'return_dict':True}
        if token_type_ids is not None:kwargs['token_type_ids']=token_type_ids
        out=self.encoder(**kwargs)
        tokens=self.dropout(out.last_hidden_state)
        pooled=self.dropout(out.last_hidden_state[:,0,:])
        return {'ner':self.ner_head(tokens),
                'category':self.category_head(pooled),
                'priority':self.priority_head(pooled)}

def joint_loss(out,batch):
    losses={
        'ner':F.cross_entropy(out['ner'].reshape(-1,len(NER_TAGS)),batch['ner_labels'].reshape(-1),ignore_index=-100),
        'category':F.cross_entropy(out['category'],batch['category_labels']),
        'priority':F.cross_entropy(out['priority'],batch['priority_labels']),
    }
    total=sum(LOSS_WEIGHTS[k]*v for k,v in losses.items())
    return total,losses

MODEL=UnifiedMARBERT(IDRISI_PATH).to(DEVICE)
print('Unified parameters:',sum(p.numel() for p in MODEL.parameters()))
print('Encoder is shared:',id(MODEL.encoder))

Unified parameters: 162260749
Encoder is shared: 133359376154544


### Metrics Used During Validation

NER is evaluated using **exact entity-level matching**. Token accuracy is secondary.

Category and priority are evaluated using Accuracy, Precision, Recall, and **Macro F1**.

The checkpoint-selection score is:

`Composite = 0.40 × NER F1 + 0.30 × Category Macro F1 + 0.30 × Priority Macro F1`

This is a model-selection metric, not an emergency-safety score.

In [39]:
def bio_spans(tags):
    spans=[];start=None
    for i,tag in enumerate(list(tags)+['O']):
        if tag=='B-LOC':
            if start is not None:spans.append((start,i))
            start=i
        elif tag=='I-LOC':
            if start is None:start=i
        else:
            if start is not None:spans.append((start,i));start=None
    return set(spans)

def score_ner(golds,preds):
    tp=fp=fn=correct=total=0
    for g,p in zip(golds,preds):
        true= bio_spans(g); estimated=bio_spans(p)
        tp+=len(true & estimated);fp+=len(estimated-true);fn+=len(true-estimated)
        correct+=sum(a==b for a,b in zip(g,p));total+=len(g)
    pr=tp/(tp+fp) if tp+fp else 0.0
    rc=tp/(tp+fn) if tp+fn else 0.0
    return {'precision':pr,'recall':rc,'f1':2*pr*rc/(pr+rc) if pr+rc else 0.0,
            'token_accuracy':correct/total if total else 0.0,'tp':tp,'fp':fp,'fn':fn}

def score_class(gold,pred,names):
    precision,recall,f1,_=precision_recall_fscore_support(gold,pred,
        labels=list(range(len(names))),average='macro',zero_division=0)
    return {'macro_f1':float(f1),'precision':float(precision),'recall':float(recall),
            'accuracy':float(accuracy_score(gold,pred))}

@torch.inference_mode()
def collect_predictions(model,loader):
    model.eval()
    ner_gold=[];ner_pred=[];cat_gold=[];cat_pred=[];pri_gold=[];pri_pred=[];pri_prob=[]
    for batch in loader:
        x={k:v.to(DEVICE) for k,v in batch.items()}
        out=model(x['input_ids'],x['attention_mask'],x.get('token_type_ids'))
        n=out['ner'].argmax(-1).cpu().tolist();labels=batch['ner_labels'].tolist()
        for pred_row,gold_row in zip(n,labels):
            keep=[(NER_TAGS[g],NER_TAGS[p]) for p,g in zip(pred_row,gold_row) if g!=-100]
            ner_gold.append([g for g,p in keep]);ner_pred.append([p for g,p in keep])
        cat_gold.extend(batch['category_labels'].tolist());cat_pred.extend(out['category'].argmax(-1).cpu().tolist())
        pri_gold.extend(batch['priority_labels'].tolist());pri_pred.extend(out['priority'].argmax(-1).cpu().tolist())
        pri_prob.extend(F.softmax(out['priority'].float(),-1).cpu().tolist())
    return {'ner_gold':ner_gold,'ner_pred':ner_pred,'cat_gold':cat_gold,'cat_pred':cat_pred,
            'pri_gold':pri_gold,'pri_pred':pri_pred,'pri_prob':pri_prob}

def eval_report(pred):
    return {'ner':score_ner(pred['ner_gold'],pred['ner_pred']),
            'category':score_class(pred['cat_gold'],pred['cat_pred'],CATEGORIES),
            'priority':score_class(pred['pri_gold'],pred['pri_pred'],PRIORITIES)}

def selection_score(scores):
    return (SELECT_WEIGHTS['ner']*scores['ner']['f1']+
            SELECT_WEIGHTS['category']*scores['category']['macro_f1']+
            SELECT_WEIGHTS['priority']*scores['priority']['macro_f1'])

# Quick invariant tests before expensive training
assert bio_spans(['B-LOC','I-LOC','O','B-LOC'])=={(0,2),(3,4)}
assert score_ner([['B-LOC','I-LOC']],[['B-LOC','I-LOC']])['f1']==1.0
print('Metric helper tests passed.')

Metric helper tests passed.


## 5. Training / Processing

Training settings:

| Parameter | Value |
|---|---:|
| Max sequence length | 192 |
| Epochs | 8 |
| Train batch size | 8 |
| Gradient accumulation | 2 |
| Effective batch size | 16 |
| Evaluation batch size | 16 |
| Encoder learning rate | `8e-6` |
| Head learning rate | `2e-5` |
| Optimizer | AdamW |
| Weight decay | 0.01 |
| Warm-up ratio | 0.10 |
| Early-stopping patience | 2 |
| Seed | 42 |
| Mixed precision | FP16 on CUDA |
| Loss weights | 1.0 / 1.0 / 1.0 |

A smaller learning rate is used for the pretrained encoder and a larger rate for the new task heads. A linear warm-up/decay scheduler is used. The best state is saved according to the validation composite.

In [40]:
from contextlib import nullcontext

encoder_params=list(MODEL.encoder.named_parameters())
head_params=list(MODEL.ner_head.parameters())+list(MODEL.category_head.parameters())+list(MODEL.priority_head.parameters())
optimizer=torch.optim.AdamW([
    {'params':[p for n,p in encoder_params if not n.endswith('bias') and 'LayerNorm.weight' not in n],
     'lr':ENCODER_LR,'weight_decay':WEIGHT_DECAY},
    {'params':[p for n,p in encoder_params if n.endswith('bias') or 'LayerNorm.weight' in n],
     'lr':ENCODER_LR,'weight_decay':0.0},
    {'params':head_params,'lr':HEAD_LR,'weight_decay':WEIGHT_DECAY},
])
updates_per_epoch=(len(TRAIN_LOADER)+GRAD_ACCUM-1)//GRAD_ACCUM
total_steps=EPOCHS*updates_per_epoch
scheduler=get_linear_schedule_with_warmup(optimizer,num_warmup_steps=max(1,int(total_steps*WARMUP_RATIO)),
                                         num_training_steps=total_steps)
SCALER=torch.amp.GradScaler('cuda',enabled=True)
BEST_FILE=OUT/'unified_best.safetensors'
HISTORY=[];best=-1;bad_epochs=0;global_updates=0

for epoch in range(1,EPOCHS+1):
    MODEL.train();optimizer.zero_grad(set_to_none=True)
    sums=Counter();num_batches=0
    for j,batch in enumerate(TRAIN_LOADER):
        batch={k:v.to(DEVICE,non_blocking=True) for k,v in batch.items()}
        with torch.autocast(device_type='cuda',dtype=torch.float16):
            outputs=MODEL(batch['input_ids'],batch['attention_mask'],batch.get('token_type_ids'))
            total,parts=joint_loss(outputs,batch)
        # Handle a smaller accumulation group at the end (same average scale).
        group_start=(j//GRAD_ACCUM)*GRAD_ACCUM
        group_size=min(GRAD_ACCUM,len(TRAIN_LOADER)-group_start)
        SCALER.scale(total/group_size).backward()
        sums['total']+=float(total.detach());num_batches+=1
        for name,value in parts.items():sums[name]+=float(value.detach())
        if (j+1)%GRAD_ACCUM==0 or (j+1)==len(TRAIN_LOADER):
            SCALER.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(MODEL.parameters(),1.0)
            scale_before = SCALER.get_scale()
            SCALER.step(optimizer)
            SCALER.update()
            # With AMP, an overflow skips optimizer.step(); don't advance LR then.
            if SCALER.get_scale() >= scale_before:
                scheduler.step()
                global_updates += 1
            optimizer.zero_grad(set_to_none=True)
    pred=collect_predictions(MODEL,VAL_LOADER)
    val=eval_report(pred)
    composite=selection_score(val)
    row={'epoch':epoch,'step':global_updates,'train_total_loss':sums['total']/num_batches,
         'train_ner_loss':sums['ner']/num_batches,
         'train_category_loss':sums['category']/num_batches,
         'train_priority_loss':sums['priority']/num_batches,
         'val_ner_f1':val['ner']['f1'],'val_category_macro_f1':val['category']['macro_f1'],
         'val_priority_macro_f1':val['priority']['macro_f1'],'val_composite':composite}
    HISTORY.append(row)
    print('Epoch',epoch,'train_loss',round(row['train_total_loss'],4),
          'NER',round(row['val_ner_f1'],4),'Category',round(row['val_category_macro_f1'],4),
          'Priority',round(row['val_priority_macro_f1'],4),'Composite',round(composite,4))
    pd.DataFrame(HISTORY).to_csv(OUT/'training_history.csv',index=False)
    if composite>best+1e-5:
        best=composite;bad_epochs=0;best_epoch=epoch
        save_file({k:v.detach().cpu().contiguous() for k,v in MODEL.state_dict().items()},str(BEST_FILE))
        print('  saved BEST epoch:',best_epoch)
    else:
        bad_epochs+=1
        if bad_epochs>=PATIENCE:
            print('Early stopping: no composite F1 improvement for',PATIENCE,'epochs')
            break

# Restore best shared encoder + three heads.
MODEL.load_state_dict(load_file(str(BEST_FILE),device='cpu'))
MODEL.to(DEVICE).eval()
TOKENIZER.save_pretrained(str(OUT/'tokenizer'))
CONFIG={'architecture':'shared MARBERT + NER/category/priority heads',
        'base_idrisi_path':str(IDRISI_PATH),'best_epoch':best_epoch,'best_val_composite':best,
        'ner_tags':NER_TAGS,'incident_category':CATEGORIES,'priority':PRIORITIES,
        'max_length':MAX_LEN,'loss_weights':LOSS_WEIGHTS,'selection_weights':SELECT_WEIGHTS,
        'learning_rates':{'encoder':ENCODER_LR,'heads':HEAD_LR},'train_batch':TRAIN_BATCH,
        'gradient_accumulation':GRAD_ACCUM,'seed':SEED,
        'base_model_config':MODEL.encoder.config.to_dict()}
(OUT/'unified_config.json').write_text(json.dumps(CONFIG,ensure_ascii=False,indent=2),encoding='utf-8')
print('Best epoch:',best_epoch,'best validation score:',round(best,4))

Epoch 1 train_loss 3.3772 NER 0.7215 Category 0.3687 Priority 0.2876 Composite 0.4855
  saved BEST epoch: 1
Epoch 2 train_loss 2.5416 NER 0.7963 Category 0.7729 Priority 0.2844 Composite 0.6357
  saved BEST epoch: 2
Epoch 3 train_loss 1.8423 NER 0.8169 Category 0.8299 Priority 0.4538 Composite 0.7119
  saved BEST epoch: 3
Epoch 4 train_loss 1.4995 NER 0.8095 Category 0.8505 Priority 0.4715 Composite 0.7204
  saved BEST epoch: 4
Epoch 5 train_loss 1.3025 NER 0.819 Category 0.8453 Priority 0.5502 Composite 0.7463
  saved BEST epoch: 5
Epoch 6 train_loss 1.1437 NER 0.7885 Category 0.8553 Priority 0.6159 Composite 0.7567
  saved BEST epoch: 6
Epoch 7 train_loss 1.0347 NER 0.8057 Category 0.8452 Priority 0.5971 Composite 0.755
Epoch 8 train_loss 0.9932 NER 0.8019 Category 0.8553 Priority 0.6064 Composite 0.7592
  saved BEST epoch: 8
Best epoch: 8 best validation score: 0.7592


### Training / Processing Observations

Training loss decreased from **3.3772** at Epoch 1 to **0.9932** at Epoch 8.

| Epoch | NER F1 | Category Macro F1 | Priority Macro F1 | Composite |
|---:|---:|---:|---:|---:|
| 1 | 0.7215 | 0.3687 | 0.2876 | 0.4855 |
| 2 | 0.7963 | 0.7729 | 0.2844 | 0.6357 |
| 3 | 0.8169 | 0.8299 | 0.4538 | 0.7119 |
| 4 | 0.8095 | 0.8505 | 0.4715 | 0.7204 |
| 5 | 0.8190 | 0.8453 | 0.5502 | 0.7463 |
| 6 | 0.7885 | 0.8553 | 0.6159 | 0.7567 |
| 7 | 0.8057 | 0.8452 | 0.5971 | 0.7550 |
| 8 | 0.8019 | 0.8553 | 0.6064 | **0.7592** |

**Best checkpoint: Epoch 8**, with validation composite `0.7592`.

The tasks improve at different rates: location extraction peaks earlier, while priority classification benefits more from later epochs. This is expected because all tasks share the same encoder.

## 6. Evaluation & Results

The final selected model is evaluated on the held-out test set only after training.

Metrics:
- **NER:** Precision, Recall, F1, token accuracy.
- **Category:** Accuracy, Precision, Recall, Macro F1, classification report, confusion matrix.
- **Priority:** Accuracy, Precision, Recall, Macro F1, classification report, confusion matrix.

Two subsets are reported:
- `test_all`: 236 reports.
- `test_real`: 62 real reports.

In [41]:
TEST_PRED=collect_predictions(MODEL,TEST_LOADER)

def subset_prediction(pred,mask):
    keys=('ner_gold','ner_pred','cat_gold','cat_pred','pri_gold','pri_pred','pri_prob')
    idx=[i for i,keep in enumerate(mask) if keep]
    return {k:[pred[k][i] for i in idx] for k in keys}

mask_real=SPLITS['test'].source.astype(str).str.lower().eq('real').to_numpy()
TEST_RESULTS={}
for subset,mask in [('test_all',np.ones(len(SPLITS['test']),dtype=bool)),('test_real',mask_real)]:
    if not mask.any():continue
    p=subset_prediction(TEST_PRED,mask)
    s=eval_report(p)
    s['category']['classification_report']=classification_report(p['cat_gold'],p['cat_pred'],
        labels=list(range(len(CATEGORIES))),target_names=CATEGORIES,output_dict=True,zero_division=0)
    s['category']['confusion_matrix']=confusion_matrix(p['cat_gold'],p['cat_pred'],
        labels=list(range(len(CATEGORIES)))).tolist()
    s['priority']['classification_report']=classification_report(p['pri_gold'],p['pri_pred'],
        labels=list(range(len(PRIORITIES))),target_names=PRIORITIES,output_dict=True,zero_division=0)
    s['priority']['confusion_matrix']=confusion_matrix(p['pri_gold'],p['pri_pred'],
        labels=list(range(len(PRIORITIES)))).tolist()
    TEST_RESULTS[subset]={'n':int(mask.sum()),**s}
    print(subset,'N=',int(mask.sum()),'NER F1=',round(s['ner']['f1'],4),
          'Category Macro F1=',round(s['category']['macro_f1'],4),
          'Priority Macro F1=',round(s['priority']['macro_f1'],4))

(OUT/'test_metrics.json').write_text(json.dumps(TEST_RESULTS,ensure_ascii=False,indent=2),encoding='utf-8')
priority_df=SPLITS['test'][['report','source','priority','incident_type']].reset_index(drop=True).copy()
priority_df['predicted_priority']=[PRIORITIES[i] for i in TEST_PRED['pri_pred']]
priority_df['predicted_probability']=[round(max(x),4) for x in TEST_PRED['pri_prob']]
priority_df['distance']=[p-g for p,g in zip(TEST_PRED['pri_pred'],TEST_PRED['pri_gold'])]
priority_df['error_type']=priority_df.distance.map(lambda n:'Correct' if n==0 else
    'Adjacent — human review REQUIRED' if abs(n)==1 else 'Major error')
priority_df['underestimated']=priority_df.distance.lt(0)
priority_df.to_csv(OUT/'priority_test_predictions.csv',index=False,encoding='utf-8-sig')
display(priority_df.loc[priority_df.distance.ne(0)].head(20))

test_all N= 236 NER F1= 0.7872 Category Macro F1= 0.8841 Priority Macro F1= 0.5897
test_real N= 62 NER F1= 0.6383 Category Macro F1= 0.8989 Priority Macro F1= 0.6745


,report,source,priority,incident_type,predicted_priority,predicted_probability,distance,error_type,underestimated
0,حادث حافلة قرب البركة وفيه عدة اصابات خطيرة ال...,generated,High,Road/Transportation,Critical,0.4891,1,Adjacent — human review REQUIRED,False
2,مركبة قلبت في جنزور غرب طرابلس وفيه زوز اصابات...,generated,High,Road/Transportation,Medium,0.3762,-1,Adjacent — human review REQUIRED,True
4,الضي يقطع ويجي كل شوية دقايق في الزاوية المركز...,generated,Low,Infrastructure/Utilities,Medium,0.7630,1,Adjacent — human review REQUIRED,False
6,دخول مياه الغدير للطوابق الارضية والمنازل بمنط...,generated,High,Flood/Severe Weather,Medium,0.6654,-1,Adjacent — human review REQUIRED,True
9,استجابة سريعة لنقل مصابين من الجفرة بواسطة الا...,real,High,People at Risk/Medical,Medium,0.3915,-1,Adjacent — human review REQUIRED,True
19,لينا خمسة ايام بدون ضي في بني وليد,generated,High,Infrastructure/Utilities,Medium,0.6589,-1,Adjacent — human review REQUIRED,True
22,انتشار مرض حاد في مركز ايواء بقصر بن غشير وعشر...,generated,Critical,People at Risk/Medical,High,0.5587,-1,Adjacent — human review REQUIRED,True
25,في سيارة ليها شهرين واقفة في شارع المعدات الطبية,generated,Low,Road/Transportation,Medium,0.5442,1,Adjacent — human review REQUIRED,False
26,خط غاز رئيسي يتسرب في الجميل والناس القريبين ط...,generated,Critical,Infrastructure/Utilities,High,0.6609,-1,Adjacent — human review REQUIRED,True
28,هبوط بسيط في الطريق عند هون وسكروا مسار واحد ل...,generated,Medium,Infrastructure/Utilities,Low,0.5265,-1,Adjacent — human review REQUIRED,True


### Location Extraction Before vs After Libya Adaptation

For a fair comparison, the original **IDRISI-only NER model** is evaluated on exactly the same Balaagh test rows.

In [42]:
@torch.inference_mode()
def idrisi_only_ner_preds(dataset):
    baseline=AutoModelForTokenClassification.from_pretrained(str(IDRISI_PATH)).to(DEVICE).eval()
    labelmap={int(i):str(x) for i,x in baseline.config.id2label.items()}
    loader=DataLoader(dataset,batch_size=EVAL_BATCH,shuffle=False,pin_memory=True)
    gold_all=[];pred_all=[]
    for b in loader:
        inp={k:b[k].to(DEVICE) for k in ('input_ids','attention_mask')}
        if 'token_type_ids' in b:inp['token_type_ids']=b['token_type_ids'].to(DEVICE)
        pred=baseline(**inp).logits.argmax(-1).cpu().tolist()
        for guess,real in zip(pred,b['ner_labels'].tolist()):
            gold_all.append([NER_TAGS[g] for g in real if g!=-100])
            pred_all.append([labelmap[p] for p,g in zip(guess,real) if g!=-100])
    baseline.cpu();del baseline
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return gold_all,pred_all

b_gold,b_pred=idrisi_only_ner_preds(DATA['test'])
COMPARISON=[]
for subset,mask in [('test_all',np.ones(len(mask_real),dtype=bool)),('test_real',mask_real)]:
    idx=[i for i,keep in enumerate(mask) if keep]
    for name,golds,preds in [('IDRISI_only',b_gold,b_pred),
                             ('IDRISI_plus_Balaagh_UNIFIED',TEST_PRED['ner_gold'],TEST_PRED['ner_pred'])]:
        m=score_ner([golds[i] for i in idx],[preds[i] for i in idx])
        COMPARISON.append({'model':name,'subset':subset,**m})
comp_df=pd.DataFrame(COMPARISON)
display(comp_df)
comp_df.to_csv(OUT/'ner_before_after.csv',index=False,encoding='utf-8-sig')

,model,subset,precision,recall,f1,token_accuracy,tp,fp,fn
0,IDRISI_only,test_all,0.481061,0.604762,0.535865,0.910078,127,137,83
1,IDRISI_plus_Balaagh_UNIFIED,test_all,0.757709,0.819048,0.787185,0.967132,172,55,38
2,IDRISI_only,test_real,0.333333,0.571429,0.421053,0.832455,36,72,27
3,IDRISI_plus_Balaagh_UNIFIED,test_real,0.576923,0.714286,0.638298,0.933614,45,33,18


### Result Interpretation

| Task | Test All | Test Real |
|---|---:|---:|
| Location NER F1 | **0.7872** | **0.6383** |
| Incident Category Macro F1 | **0.8841** | **0.8989** |
| Priority Macro F1 | **0.5897** | **0.6745** |

**Incident category** is the strongest task.

**Priority classification** remains the most difficult because neighboring severity levels are semantically close and some reports omit important severity-driving facts.

Location adaptation improved substantially:

| Model | Test All NER F1 | Test Real NER F1 |
|---|---:|---:|
| IDRISI-only | 0.5359 | 0.4211 |
| IDRISI + Balaagh Unified | **0.7872** | **0.6383** |

This corresponds to approximately **+25.13 percentage points** on the full test set and **+21.72 points** on real test reports.

## 7. Error Analysis

Priority errors are analyzed using the ordered classes:

`Low → Medium → High → Critical`

- Difference of one level: **Adjacent — human review required**
- Difference of two or more levels: **Major error**

An adjacent prediction is not automatically acceptable. Underestimating a `Critical` event is especially important and must be reviewed.

In [ ]:
# Priority error analysis
print("Priority error types:")
display(priority_df["error_type"].value_counts(dropna=False).to_frame("count"))

print("\nUnderestimated cases:")
display(
    priority_df.loc[
        priority_df["underestimated"],
        ["report", "source", "priority", "predicted_priority",
         "predicted_probability", "distance", "error_type"]
    ].head(30)
)

print("\nMajor errors:")
display(
    priority_df.loc[
        priority_df["error_type"].eq("Major error"),
        ["report", "source", "priority", "predicted_priority",
         "predicted_probability", "distance"]
    ].head(30)
)

Important observations:
- Missing information must not be interpreted as zero risk.
- Real reports can be vague or incomplete.
- Synthetic patterns may be easier to learn than naturally occurring reports.
- Gold priority labels should be reviewed independently rather than changed to match predictions.
- High-stakes underestimation, especially `Critical → High/Medium`, requires special attention.

## 8. Sample Predictions / Outputs

The inference function below takes one Arabic/Libyan report and returns:
- all explicit location mentions,
- incident category and class scores,
- priority and class scores.

The NER output represents locations explicitly mentioned in the report; it does not infer an unstated incident location.

In [43]:
@torch.inference_mode()
def predict_report(text):
    if not str(text).strip():raise ValueError('Please enter a report.')
    MODEL.eval()
    enc=TOKENIZER(str(text),return_offsets_mapping=True,truncation=True,max_length=MAX_LEN,return_tensors='pt')
    offsets=enc.pop('offset_mapping')[0].tolist()
    inp={k:v.to(DEVICE) for k,v in enc.items()}
    out=MODEL(inp['input_ids'],inp['attention_mask'],inp.get('token_type_ids'))
    tag_probs=F.softmax(out['ner'][0].float(),dim=-1)
    tag_ids=tag_probs.argmax(-1).cpu().tolist()
    locations=[];current=None
    for idx,(st,en) in enumerate(offsets):
        if st==en:continue
        tag=NER_TAGS[tag_ids[idx]]
        if tag=='O':
            if current is not None:locations.append(current);current=None
        elif tag=='B-LOC' or current is None:
            if current is not None:locations.append(current)
            current={'start':st,'end':en,'confidences':[float(tag_probs[idx,tag_ids[idx]])]}
        else:
            current['end']=en;current['confidences'].append(float(tag_probs[idx,tag_ids[idx]]))
    if current is not None:locations.append(current)
    loc=[{'text':str(text)[x['start']:x['end']], 'start':x['start'],'end':x['end'],
          'confidence':round(min(x['confidences']),4)} for x in locations]
    cp=F.softmax(out['category'][0].float(),-1).cpu().numpy()
    pp=F.softmax(out['priority'][0].float(),-1).cpu().numpy()
    return {'report':str(text),'locations':loc,
            'incident_category':CATEGORIES[int(cp.argmax())],
            'category_scores':{k:round(float(cp[i]),4) for i,k in enumerate(CATEGORIES)},
            'priority':PRIORITIES[int(pp.argmax())],
            'priority_scores':{k:round(float(pp[i]),4) for i,k in enumerate(PRIORITIES)},
            'warning':'Model output only; not an emergency dispatch decision.'}

print(json.dumps(predict_report('حادث في تاجوراء والسائق محاصر داخل السيارة، نبو إسعاف بسرعة.'),
                 ensure_ascii=False,indent=2))

try:
    import ipywidgets as widgets
    area=widgets.Textarea(value='فيه سيول في تاجوراء والطريق مسكر.',
                          description='بلاغ:',layout=widgets.Layout(width='90%',height='100px'))
    button=widgets.Button(description='تحليل البلاغ',button_style='success')
    result=widgets.Output()
    def on_click(_):
        with result:
            result.clear_output()
            print(json.dumps(predict_report(area.value),ensure_ascii=False,indent=2))
    button.on_click(on_click)
    display(widgets.VBox([area,button,result]))
except ImportError:
    print('ipywidgets unavailable; call predict_report(text) directly.')

{
  "report": "حادث في تاجوراء والسائق محاصر داخل السيارة، نبو إسعاف بسرعة.",
  "locations": [
    {
      "text": "تاجوراء",
      "start": 8,
      "end": 15,
      "confidence": 0.973
    }
  ],
  "incident_category": "Road/Transportation",
  "category_scores": {
    "Fire/Explosion": 0.0204,
    "Flood/Severe Weather": 0.0109,
    "Infrastructure/Utilities": 0.0028,
    "Road/Transportation": 0.8956,
    "People at Risk/Medical": 0.0665,
    "Other": 0.0039
  },
  "priority": "Critical",
  "priority_scores": {
    "Low": 0.0324,
    "Medium": 0.0947,
    "High": 0.3231,
    "Critical": 0.5497
  },
  "warning": "Model output only; not an emergency dispatch decision."
}


### Saved Outputs

The final archive contains the best unified weights, tokenizer, configuration, training history, test metrics, priority predictions, and NER before/after comparison.

In [44]:
import zipfile
ARCHIVE=Path('/kaggle/working/crisislens_unified_three_tasks.zip')
with zipfile.ZipFile(ARCHIVE,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=3) as z:
    for name in ['unified_best.safetensors','unified_config.json','training_history.csv',
                 'test_metrics.json','priority_test_predictions.csv','ner_before_after.csv']:
        path=OUT/name
        if not path.exists():raise FileNotFoundError('Missing output '+str(path))
        z.write(path,'crisislens_unified/'+name)
    for path in sorted((OUT/'tokenizer').rglob('*')):
        if path.is_file():z.write(path,'crisislens_unified/tokenizer/'+path.name)
print('Saved:',ARCHIVE)
print('Size (MB):',round(ARCHIVE.stat().st_size/1e6,1))

Saved: /kaggle/working/crisislens_unified_three_tasks.zip
Size (MB): 605.5


## 9. Conclusion

This notebook implemented a **single jointly trained MARBERT model** for CrisisLens Libya.

Main findings:
- Incident category reached **0.8841 Macro F1** on the full test set and **0.8989** on real reports.
- Location extraction reached **0.7872 F1**, substantially improving over the IDRISI-only baseline.
- Priority classification reached **0.5897 Macro F1** overall and **0.6745** on the real subset, making it the main area requiring further improvement.

Key limitations:
- Only 62 real reports are present in the real-only test subset.
- The dataset also contains generated reports, so template effects may influence results.
- Priority labels remain difficult and should continue to be audited.
- Location extraction identifies explicit mentions but does not yet perform geographic entity linking or coordinate resolution.
- Softmax scores are model confidence values and must not be treated as guarantees of emergency safety.

Overall, unified multi-task training successfully combines location extraction and crisis classification in one MARBERT network, while showing that future work should focus especially on priority-label quality and evaluation on a larger independently reviewed real-world Libyan test set.